In [7]:
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.FashionMNIST(
    root="./resources/data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.FashionMNIST(
    root="./resources/data",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1000,
    shuffle=False
)

In [8]:
import torch.nn as nn

class LeNet5(nn.Module):

    def __init__(self):
        super(LeNet5, self).__init__()

        self.conv1 = nn.Conv2d(1, 6, kernel_size=5, stride=1, padding=2)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv2 = nn.Conv2d(6, 16, kernel_size=5)

        self.fc1 = nn.Linear(16*5*5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

        self.relu = nn.ReLU()

    def forward(self, x):

        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))

        x = x.view(x.size(0), -1)

        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)

        return x

In [9]:
import torch

def check_shape():

    model = LeNet5()

    #random
    x = torch.randn(1,1,28,28)

    print("Input:", x.shape)

    x = model.conv1(x)
    print("Conv1:", x.shape)

    x = model.relu(x)
    x = model.pool(x)
    print("Pool1:", x.shape)

    x = model.conv2(x)
    print("Conv2:", x.shape)

    x = model.relu(x)
    x = model.pool(x)
    print("Pool2:", x.shape)

    assert x.shape == (1,16,5,5)

    x = x.view(1,-1)
    print("Flatten:", x.shape)

    assert x.shape == (1,400)

check_shape()

Input: torch.Size([1, 1, 28, 28])
Conv1: torch.Size([1, 6, 28, 28])
Pool1: torch.Size([1, 6, 14, 14])
Conv2: torch.Size([1, 16, 10, 10])
Pool2: torch.Size([1, 16, 5, 5])
Flatten: torch.Size([1, 400])


In [10]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cpu


In [11]:
import torch.optim as optim

model = LeNet5().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

def train(model, loader):

    model.train()

    total_loss = 0

    for inputs, labels in loader:

        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

def test(model, loader):

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for inputs, labels in loader:

            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)

            correct += (predicted == labels).sum().item()

    accuracy = correct / total

    return accuracy

In [12]:
epochs = 10

for epoch in range(epochs):

    train_loss = train(model, train_loader)

    test_acc = test(model, test_loader)

    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Test Acc: {test_acc:.4f}")

Epoch 1/10 | Train Loss: 0.6053 | Test Acc: 0.8371
Epoch 2/10 | Train Loss: 0.3838 | Test Acc: 0.8666
Epoch 3/10 | Train Loss: 0.3250 | Test Acc: 0.8829
Epoch 4/10 | Train Loss: 0.2961 | Test Acc: 0.8847
Epoch 5/10 | Train Loss: 0.2751 | Test Acc: 0.8902
Epoch 6/10 | Train Loss: 0.2556 | Test Acc: 0.8922
Epoch 7/10 | Train Loss: 0.2396 | Test Acc: 0.8970
Epoch 8/10 | Train Loss: 0.2269 | Test Acc: 0.8895
Epoch 9/10 | Train Loss: 0.2155 | Test Acc: 0.8918
Epoch 10/10 | Train Loss: 0.2025 | Test Acc: 0.8979
